# Agent: impact

Develop and test **`agentic_scd.agents.impact.impact_node`** in isolation.

## What this agent does

```mermaid
flowchart LR
    UP["classifications<br/>Classification[]"]:::faded --> B1
    subgraph B["impact_node (per classification)"]
        B1["read category"] --> B2["IMPACT_BY_CATEGORY lookup"]
        B2 --> B3{"category known?"}
        B3 -->|yes| B4["affected suppliers / lanes / facilities"]
        B3 -->|no| B5["'other' default entities"]
    end
    B4 --> D["impacts<br/>ImpactMap[]"]
    B5 --> D
    D --> DOWN["downstream: simulate · recommend"]:::faded
    classDef faded fill:#eee,stroke:#bbb,color:#888;
```

**State contract**

- **Reads:** `classifications` (list of `Classification`)
- **Writes:** `impacts` (list of `ImpactMap`: signal_id, affected_entities)
- **Fallback / degradation:** unknown category → the `'other'` entity list

**Phase 4** replaces this hard-coded lookup with RAG over the internal supply-chain knowledge base (Chroma), behind the same `impact_node` signature.

## Is the DB up? (optional)

In [ ]:
# Optional: this agent runs fine offline on synthetic sample state. This snippet just
# reports whether the live DB is reachable (Setup section of 00_orchestration brings
# it up).
from agentic_scd.devtools import db_status

status = db_status()
print(status.detail)
if not status:
    print("Proceeding offline with synthetic sample state — fine for iterating here.")

## Build a representative input state

In [ ]:
from agentic_scd.agents.classify import classify_node
from agentic_scd.devtools import sample_state

# impact reads `classifications`, so run classify first to build the input state.
state = sample_state(count=2)
state.update(classify_node(state))
for c in state["classifications"]:
    print(f"{c.category:>16}  risk={c.risk_score:.2f}")

## Call `impact_node` in isolation

In [ ]:
from agentic_scd.agents.impact import impact_node

state.update(impact_node(state))
for im in state["impacts"]:
    print(f"{im.signal_id[:8]}  ->  {', '.join(im.affected_entities)}")

## Iterate here

This is your dev surface: tweak the input above, re-run, and watch `impact_node`'s output change. When you deepen this agent in its phase, keep the node signature the same so the rest of the graph is unaffected.